## Concept 3 — Few-Shot Prompting

### What is it?
Instead of only telling the LLM what to do, you **show it examples** of input → output pairs.
The LLM learns the exact pattern from demonstrations — format, tone, length — far more reliably than from instructions alone.

### Pipeline
```
[System] + [Example 1: Human→AI] + [Example 2: Human→AI] + [Human: real query] → LLM → Answer
```

### Limitation (what Concept 4 fixes)
LLM forgets everything between calls — no memory of previous turns.
→ Concept 4 adds MessagesPlaceholder to inject conversation history.

In [ ]:
print('Hi')

In [ ]:
import sys
sys.path.insert(0, '.')

from PromptUtils import get_llm, run_inputs
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = get_llm(temperature=0.0)
print('LLM ready')

### Step 1 — Without Few-Shot (Baseline)
Instructions alone — LLM may produce inconsistent output format.

In [ ]:
no_shot_prompt = ChatPromptTemplate.from_messages([
    ('system', 'Classify the sentiment. Reply with only: POSITIVE, NEGATIVE, or NEUTRAL.'),
    ('human',  '{text}'),
])
no_shot_chain = no_shot_prompt | llm | StrOutputParser()

test_texts = [
    'I absolutely love this product!',
    'Terrible quality, broke on day one.',
    'It works as expected.',
]

print('Without few-shot:')
for t in test_texts:
    out = no_shot_chain.invoke({'text': t})
    print(f'  "{t}" → {out}')

### Step 2 — Define Examples
Each example is a dict with keys matching the example template's input variables.

In [ ]:
examples = [
    {'input': 'I love this!',          'output': 'POSITIVE'},
    {'input': 'Terrible product.',     'output': 'NEGATIVE'},
    {'input': 'It is okay I guess.',   'output': 'NEUTRAL'},
    {'input': 'Best purchase ever!',   'output': 'POSITIVE'},
    {'input': 'Would not recommend.',  'output': 'NEGATIVE'},
]
print(f'{len(examples)} examples defined')

### Step 3 — FewShotChatMessagePromptTemplate
Wraps examples into alternating Human/AI message pairs automatically.

In [ ]:
example_prompt = ChatPromptTemplate.from_messages([
    ('human', '{input}'),
    ('ai',    '{output}'),
])

few_shot_prompt = FewShotChatMessagePromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
)

# Inspect what the few-shot block looks like
print('Few-shot messages:')
for msg in few_shot_prompt.format_messages():
    print(f'  [{msg.type}] {msg.content}')

### Step 4 — Full Few-Shot Chain
Combine system + few-shot examples + real query into one prompt.

In [ ]:
full_prompt = ChatPromptTemplate.from_messages([
    ('system', 'Classify sentiment. Reply with only one word: POSITIVE, NEGATIVE, or NEUTRAL.'),
    few_shot_prompt,
    ('human', '{input}'),
])

few_shot_chain = full_prompt | llm | StrOutputParser()

print('With few-shot:')
for t in test_texts:
    out = few_shot_chain.invoke({'input': t})
    print(f'  "{t}" → {out}')

### Bonus — Few-Shot for a Different Task: Code Translation
Few-shot works for any pattern — not just classification.

In [ ]:
code_examples = [
    {'input': 'x = [1,2,3]\nfor i in x: print(i)',
     'output': 'x = [1,2,3]\nx.forEach(i => console.log(i))'},
    {'input': 'def add(a, b):\n    return a + b',
     'output': 'function add(a, b) {\n    return a + b;\n}'},
]

code_example_prompt = ChatPromptTemplate.from_messages([
    ('human', 'Python:\n{input}'),
    ('ai',    'JavaScript:\n{output}'),
])

code_few_shot = FewShotChatMessagePromptTemplate(
    examples=code_examples,
    example_prompt=code_example_prompt,
)

code_prompt = ChatPromptTemplate.from_messages([
    ('system', 'Translate Python code to JavaScript. Output only the JavaScript code.'),
    code_few_shot,
    ('human', 'Python:\n{input}'),
])
code_chain = code_prompt | llm | StrOutputParser()

py_code = 'numbers = [1,2,3,4,5]\nresult = [n*2 for n in numbers]'
print(f'Python:\n{py_code}\n')
print(f'JavaScript:\n{code_chain.invoke({"input": py_code})}')

### Full Function

In [ ]:
def few_shot_sentiment(text):
    return few_shot_chain.invoke({'input': text})

# Quick test
custom_tests = [
    'This exceeded all my expectations.',
    'Complete waste of money.',
    'Nothing special about it.',
]

print('Sentiment Classification (Few-Shot):')
for t in custom_tests:
    print(f'  "{t}" → {few_shot_sentiment(t)}')